# Calculator-SNOBOL4

An arithmetic calculator whose whole grammar is one **pattern**. A SNOBOL4 pattern can call a function while it matches: each number pushes itself on a stack, each operator pops two values and pushes the result, and the end of a line prints the answer. This is the classic SNOBOL4 calculator. Run the cells in order with Run All, or step through it with the debugger.

In [ ]:
        DEFINE('EMIT()')
        DEFINE('PSH()')
        DEFINE('DRF(nm)')
        DEFINE('ADD()')
        DEFINE('SUB()')
        DEFINE('MUL()')
        DEFINE('DIV()')
        DEFINE('NEG()')                         :(FNS_end)
EMIT    OUTPUT = DRF(S[1])
        sp = 0
        EMIT = .dm                              :(NRETURN)

PSH     PSH = .S[sp = sp + 1]                   :(NRETURN)
DRF     DRF = nm
        nm ? POS(0) ANY(&LCASE) RPOS(0)         :F(RETURN)
        DRF = vars[nm]                          :(RETURN)
ADD     S[sp = sp - 1] = DRF(S[sp]) + DRF(S[sp + 1])
        ADD = .dm                               :(NRETURN)
SUB     S[sp = sp - 1] = DRF(S[sp]) - DRF(S[sp + 1])
        SUB = .dm                               :(NRETURN)
MUL     S[sp = sp - 1] = DRF(S[sp]) * DRF(S[sp + 1])
        MUL = .dm                               :(NRETURN)
DIV     S[sp = sp - 1] = DRF(S[sp]) / DRF(S[sp + 1])
        DIV = .dm                               :(NRETURN)
NEG     S[sp] = -DRF(S[sp])
        NEG = .dm                               :(NRETURN)
FNS_end

The grammar: a number or a letter is an atom, a sign applies to a factor, `*` and `/` bind tighter than `+` and `-`, and parentheses nest. The patterns `T` and `X` call themselves, and every `*PSH()`, `*ADD()` and so on is a function the match runs as it goes.

In [ ]:
        S = ARRAY(65536)
        V = ANY(&LCASE) . *PSH()
        I = SPAN('0123456789') . *PSH()
        A = V | I | FENCE('(' *X ')')
        F = A | FENCE('+' *F) | FENCE('-' *F . *NEG())
        T = F ( FENCE('*' *T . *MUL()) | FENCE('/' *T . *DIV()) | '' )
        X = T ( FENCE('+' *X . *ADD()) | FENCE('-' *X . *SUB()) | '' )
        eol = CHAR(10) FENCE
        C = POS(0) ARBNO(X eol . *EMIT()) RPOS(0)

Now run it. The variables `x`, `y` and `z` are 1, 2 and 3; any other letter is 0. The expressions come from `calculator.in` beside this notebook, one per line, and each prints its value.

In [ ]:
        &TRIM = 0
        vars = TABLE()
        vars['x'] = 1
        vars['y'] = 2
        vars['z'] = 3
        INPUT(.INPUT, 9, 'calculator.in')
rd      line = INPUT                                      :F(go)
        src = src line CHAR(10)                           :(rd)
go      src C                                             :S(done)F(bad)
bad     OUTPUT = 'Boo!'
done